# MSK abnormality detection (MURA) - Colab quickstart
**Educational research prototype - not a clinical tool.**

Runtime > Change runtime type > GPU. Upload the project folder as `mura_msk.zip` and your MURA-v1.1 zip to Google Drive first (MURA must be downloaded from the official Stanford page).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# EDIT these two paths
PROJECT_ZIP = '/content/drive/MyDrive/mura_msk.zip'
MURA_ZIP    = '/content/drive/MyDrive/MURA-v1.1.zip'

!unzip -q -o $PROJECT_ZIP -d /content/
!mkdir -p /content/data && unzip -q -o $MURA_ZIP -d /content/data/
!ls /content/data/MURA-v1.1 | head

In [ ]:
%cd /content/mura_msk
!pip install -q opencv-python-headless scikit-learn scipy pandas matplotlib pillow streamlit
import os
os.environ['MURA_ROOT']   = '/content/data/MURA-v1.1'
os.environ['MURA_OUTPUT'] = '/content/drive/MyDrive/mura_outputs'   # persists across disconnects

In [ ]:
# Optional: quick end-to-end check on tiny synthetic data (metrics meaningless)
# !bash scripts/smoke_test.sh

In [ ]:
# 1) Hyper-parameter search (reduce --trials if time is short)
!python -m src.tune --model baseline     --trials 8 --subset 0.25 --trial_epochs 8
!python -m src.tune --model efficientnet --trials 8 --subset 0.25 --trial_epochs 5

In [ ]:
# 2) Final training
!python -m src.train --model baseline     --run_name baseline     --epochs 25 --params $MURA_OUTPUT/tuning_baseline/best_params.json
!python -m src.train --model efficientnet --run_name efficientnet --epochs 15 --params $MURA_OUTPUT/tuning_efficientnet/best_params.json

In [ ]:
# 3) Evaluation, comparison, Grad-CAM, insights
!python -m src.evaluate --run_name baseline
!python -m src.evaluate --run_name efficientnet
!python -m src.compare
!python -m src.gradcam --run_name efficientnet --n 4
!python -m src.insights

In [ ]:
from IPython.display import Image, Markdown, display
O = os.environ['MURA_OUTPUT']
display(Image(f'{O}/comparison/metrics_comparison.png'))
display(Image(f'{O}/comparison/roc_comparison.png'))
display(Image(f'{O}/efficientnet/gradcam_examples.png'))
display(Markdown(open(f'{O}/insights.md').read()))

## Streamlit in Colab
Colab has no public port. Use a tunnel:

In [ ]:
!npm install -q localtunnel
!streamlit run app/streamlit_app.py &>/content/streamlit.log &
!curl -s ipv4.icanhazip.com   # tunnel password = this IP
!npx localtunnel --port 8501